In [45]:
import sys
from pathlib import Path

# Add the src directory to Python path
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR
import pandas as pd
import requests
from datetime import datetime
import time
import json

In [46]:
# Load the market blobs
blobs_file = INTERIM_DATA_DIR / 'musk_polymarket_blobs.json'

with open(blobs_file, 'r') as f:
    musk_blobs = json.load(f)

print(f"Loaded {len(musk_blobs)} market blobs")
print(f"\nSample market:")
print(f"  Slug: {musk_blobs[0]['slug']}")
print(f"  Condition ID: {musk_blobs[0]['condition_id']}")
print(f"  Number of outcome markets: {len(musk_blobs[0].get('markets', []))}")

Loaded 89 market blobs

Sample market:
  Slug: of-elon-musk-tweets-between-may-31-and-june-7
  Condition ID: None
  Number of outcome markets: 8


In [47]:
# Function to fetch timeseries data for a specific token/market outcome
def get_market_timeseries(token_id, debug=False):
    """
    Fetch timeseries price data for a specific market token.
    
    Args:
        token_id: The token ID (asset ID) for the outcome
        debug: Print debug information
    
    Returns:
        list: Timeseries data points
    """
    try:
        url = "https://clob.polymarket.com/prices-history"
        
        # The token_id IS the market/asset ID - use it directly
        params = {
            'market': token_id,
            'interval': 'max',
            'fidelity': 60
        }
        
        if debug:
            print(f"      DEBUG: Token/Asset ID: {token_id}")
            print(f"      DEBUG: URL: {url}")
            print(f"      DEBUG: Params: {params}")
        
        response = requests.get(url, params=params, timeout=30)
        
        if debug:
            print(f"      DEBUG: Status: {response.status_code}")
            print(f"      DEBUG: Response text: {response.text[:500]}")
        
        if response.status_code == 200:
            data = response.json()
            history = data.get('history', [])
            
            if debug:
                print(f"      DEBUG: History length: {len(history)}")
                if len(history) > 0:
                    print(f"      DEBUG: Sample data point: {history[0]}")
            
            return history
        else:
            if debug:
                print(f"      DEBUG: Failed with status {response.status_code}")
            return []
        
    except Exception as e:
        if debug:
            print(f"      DEBUG: Exception: {e}")
        return []

print("Function defined successfully")

Function defined successfully


In [48]:
# Create directory for timeseries data
timeseries_dir = INTERIM_DATA_DIR / 'polymarket_timeseries'
timeseries_dir.mkdir(exist_ok=True)

print(f"Timeseries data will be saved to: {timeseries_dir}")

Timeseries data will be saved to: /Users/vince/Developer/indian_pizza_machine/backend/data/interim/polymarket_timeseries


In [49]:
# Since timeseries data is not available, extract available market metadata instead
processed_count = 0
all_market_data = []

print(f"NOTE: Timeseries data not available from API for these markets.")
print(f"Extracting available market metadata instead...\n")

for i, blob in enumerate(musk_blobs, 1):
    slug = blob['slug']
    markets = blob.get('markets', [])
    
    print(f"[{i}/{len(musk_blobs)}] Processing: {slug} ({len(markets)} outcomes)")
    
    if not markets:
        continue
    
    for market in markets:
        # Extract all available metadata
        try:
            token_ids = json.loads(market.get('clobTokenIds', '[]'))
        except:
            token_ids = []
        
        try:
            outcome_prices = json.loads(market.get('outcomePrices', '[]'))
        except:
            outcome_prices = []
        
        market_data = {
            'parent_slug': slug,
            'market_id': market.get('id'),
            'question': market.get('question'),
            'condition_id': market.get('conditionId'),
            'slug': market.get('slug'),
            'end_date': market.get('endDate'),
            'closed': market.get('closed'),
            'active': market.get('active'),
            'volume': float(market.get('volume', 0)),
            'volume_num': float(market.get('volumeNum', 0)),
            'clob_token_id': token_ids[0] if token_ids else None,
            'outcome_prices': outcome_prices,
            'last_trade_price': market.get('lastTradePrice'),
            'best_bid': market.get('bestBid'),
            'best_ask': market.get('bestAsk'),
            'group_item_title': market.get('groupItemTitle'),
            'created_at': market.get('createdAt'),
            'updated_at': market.get('updatedAt'),
            'closed_time': market.get('closedTime'),
            'uma_resolution_status': market.get('umaResolutionStatus'),
        }
        
        all_market_data.append(market_data)
    
    processed_count += 1

# Create DataFrame and save
df = pd.DataFrame(all_market_data)

# Convert date columns
date_cols = ['end_date', 'created_at', 'updated_at', 'closed_time']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Save to parquet
output_file = INTERIM_DATA_DIR / 'musk_polymarket_outcomes_metadata.parquet'
df.to_parquet(output_file, index=False)

print(f"\n{'='*70}")
print(f"Summary:")
print(f"  Processed markets: {processed_count}/{len(musk_blobs)}")
print(f"  Total outcomes: {len(df)}")
print(f"  Saved to: {output_file.name}")
print(f"{'='*70}")

NOTE: Timeseries data not available from API for these markets.
Extracting available market metadata instead...

[1/89] Processing: of-elon-musk-tweets-between-may-31-and-june-7 (8 outcomes)
[2/89] Processing: of-elon-musk-tweets-june-7-14 (8 outcomes)
[3/89] Processing: elon-musk-of-tweets-june-14-21 (8 outcomes)
[4/89] Processing: elon-musk-of-tweets-june-21-28 (8 outcomes)
[5/89] Processing: elon-musk-of-tweets-june-28-july-5 (8 outcomes)
[6/89] Processing: elon-musk-of-tweets-july-5-july-12 (8 outcomes)
[7/89] Processing: elon-musk-of-tweets-july-12-july-19 (8 outcomes)
[8/89] Processing: elon-musk-of-tweets-july-19-26 (8 outcomes)
[9/89] Processing: elon-musk-of-tweets-august-2-9 (9 outcomes)
[10/89] Processing: elon-musk-of-tweets-august-9-16 (9 outcomes)
[11/89] Processing: elon-musk-of-tweets-august-16-23 (10 outcomes)
[12/89] Processing: elon-musk-of-tweets-august-23-30 (11 outcomes)
[13/89] Processing: elon-musk-of-tweets-sept-6-13 (10 outcomes)
[14/89] Processing: elon-musk-

In [50]:
# Display summary statistics
print(f"\nDataset overview:")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSample data:")
display(df.head(10))

print(f"\nValue counts by parent market:")
print(df['parent_slug'].value_counts().head(10))


Dataset overview:
Shape: (1519, 20)

Columns: ['parent_slug', 'market_id', 'question', 'condition_id', 'slug', 'end_date', 'closed', 'active', 'volume', 'volume_num', 'clob_token_id', 'outcome_prices', 'last_trade_price', 'best_bid', 'best_ask', 'group_item_title', 'created_at', 'updated_at', 'closed_time', 'uma_resolution_status']

Sample data:


,parent_slug,market_id,question,condition_id,slug,end_date,closed,active,volume,volume_num,clob_token_id,outcome_prices,last_trade_price,best_bid,best_ask,group_item_title,created_at,updated_at,closed_time,uma_resolution_status
0,of-elon-musk-tweets-between-may-31-and-june-7,501920,Will Elon tweet less than 40 times?,0xef16e1f4617e12af17bb2acfd5302d146f73f51e8c4b...,will-elon-tweet-less-than-40-times,2024-06-07 12:00:00+00:00,True,True,29203.475858,29203.48,9350411719202949898954130748661256865836322107...,"[0, 1]",0.0,0.0,1.0,<40,2024-05-30 20:03:56.746814+00:00,2024-06-06 05:36:13.092132+00:00,2024-06-05 08:47:18+00:00,resolved
1,of-elon-musk-tweets-between-may-31-and-june-7,501921,Will Elon tweet between 40 and 49 times?,0xf81a8cc28442761b1509878d51c9ba1e1b48e57060cc...,will-elon-tweet-between-40-and-49-times,2024-06-07 12:00:00+00:00,True,True,15341.914399,15341.91,1898740429126848037210087796349461761048501340...,"[0, 1]",0.0,0.0,1.0,40-49,2024-05-30 20:05:17.722550+00:00,2024-06-06 22:16:13.361852+00:00,2024-06-06 02:07:23+00:00,resolved
2,of-elon-musk-tweets-between-may-31-and-june-7,501922,Will Elon tweet between 50 and 59 times?,0x291eeb9ed93003f041e8ce45d5a02005718395f57a29...,will-elon-tweet-between-50-and-59-times,2024-06-07 12:00:00+00:00,True,True,11330.707858,11330.71,3846331443052953110056297997279902256504905769...,"[0, 1]",0.0,0.0,1.0,50-59,2024-05-30 20:07:17.246905+00:00,2024-06-07 14:11:16.867279+00:00,2024-06-06 16:42:09+00:00,resolved
3,of-elon-musk-tweets-between-may-31-and-june-7,501923,Will Elon tweet between 60 and 69 times?,0xf2d28d566aa80bf179e49200597797de7d7837a27915...,will-elon-tweet-between-60-and-69-times,2024-06-07 12:00:00+00:00,True,True,11296.601206,11296.60,3692535710860776302039826056344735112906573801...,"[0, 1]",0.0,0.0,1.0,60-69,2024-05-30 20:07:47.591889+00:00,2024-06-07 16:26:15.030569+00:00,2024-06-06 19:59:59+00:00,resolved
4,of-elon-musk-tweets-between-may-31-and-june-7,501924,Will Elon tweet between 70 and 79 times?,0xe9051fb4364da329a91a59a487800289cb73486e99c6...,will-elon-tweet-between-70-and-79-times,2024-06-07 12:00:00+00:00,True,True,9358.420021,9358.42,7690693134338655820231068121589108940079872282...,"[0, 1]",0.0,0.0,1.0,70-79,2024-05-30 20:08:27.577420+00:00,2024-06-07 22:16:16.801521+00:00,2024-06-07 01:47:01+00:00,resolved
5,of-elon-musk-tweets-between-may-31-and-june-7,501925,Will Elon tweet between 80 and 89 times?,0xd9440c9bc6aa38abbcbe95c6be83b882369b0c87cd86...,will-elon-tweet-between-80-and-89-times,2024-06-07 12:00:00+00:00,True,True,17403.208175,17403.21,3909358302225952703240386268982431435543235425...,"[1, 0]",0.0,0.0,1.0,80-89,2024-05-30 20:08:49.223613+00:00,2024-06-08 16:11:15.434557+00:00,2024-06-07 19:16:58+00:00,resolved
6,of-elon-musk-tweets-between-may-31-and-june-7,501926,Will Elon tweet between 90 and 99 times?,0xd06ac359f4dbecafb74d71bc7aa918b33fee9d26d5dc...,will-elon-tweet-between-90-and-99-times,2024-06-07 12:00:00+00:00,True,True,21626.611409,21626.61,2658142587880723925205449678884179404419092872...,"[0, 1]",0.0,0.0,1.0,90-99,2024-05-30 20:09:13.271843+00:00,2024-06-08 15:46:12.343431+00:00,2024-06-07 19:16:52+00:00,resolved
7,of-elon-musk-tweets-between-may-31-and-june-7,501928,Will Elon tweet 100 or more times?,0xd74bfed984c5d835b7f2ef83555a4c40c4785b172852...,will-elon-tweet-100-or-more-times,2024-06-07 12:00:00+00:00,True,True,20508.050113,20508.05,2905318633445867821027592929291935230556508853...,"[0, 1]",0.0,0.0,1.0,100+,2024-05-30 20:09:57.706594+00:00,2024-06-08 14:06:12.795631+00:00,2024-06-07 19:16:48+00:00,resolved
8,of-elon-musk-tweets-june-7-14,502201,Will Elon post less than 40 times?,0x0051ea6db91583770e996964a086c1817c4527d9307c...,will-elon-post-less-than-40-times,2024-06-14 12:00:00+00:00,True,True,31056.663280,31056.66,4365797916957621393684200261771937815060430482...,"[0, 1]",0.0,0.0,1.0,<40,2024-06-07 16:02:51.968895+00:00,2024-06-13 17:34:59.708207+00:00,2024-06-11 22:20:22+00:00,resolved
9,of-elon-musk-tweets-june-7-14


Value counts by parent market:
parent_slug
elon-musk-of-tweets-january-16-january-23        38
elon-musk-of-tweets-january-23-january-30        38
elon-musk-of-tweets-january-30-february-6        38
elon-musk-of-tweets-september-12-september-19    30
elon-musk-of-tweets-september-19-september-26    30
elon-musk-of-tweets-october-24-october-31        30
elon-musk-of-tweets-december-12-december-19      30
elon-musk-of-tweets-december-19-december-26      30
elon-musk-of-tweets-december-26-january-2        30
elon-musk-of-tweets-january-2-january-9          30
Name: count, dtype: int64


In [51]:
# Load and inspect the saved metadata file
metadata_file = INTERIM_DATA_DIR / 'musk_polymarket_outcomes_metadata.parquet'

if metadata_file.exists():
    df_loaded = pd.read_parquet(metadata_file)
    
    print(f"Loaded: {metadata_file.name}")
    print(f"Shape: {df_loaded.shape}")
    print(f"\nData types:")
    print(df_loaded.dtypes)
    
    print(f"\n\nBasic statistics:")
    print(df_loaded[['volume', 'volume_num']].describe())
    
    print(f"\n\nResolution status distribution:")
    print(df_loaded['uma_resolution_status'].value_counts())
    
    print(f"\n\nSample records:")
    display(df_loaded.sample(min(5, len(df_loaded))))
else:
    print("Metadata file not found. Run the previous cell first.")

Loaded: musk_polymarket_outcomes_metadata.parquet
Shape: (1519, 20)

Data types:
parent_slug                              str
market_id                                str
question                                 str
condition_id                             str
slug                                     str
end_date                 datetime64[us, UTC]
closed                                  bool
active                                  bool
volume                               float64
volume_num                           float64
clob_token_id                            str
outcome_prices                        object
last_trade_price                     float64
best_bid                             float64
best_ask                             float64
group_item_title                         str
created_at               datetime64[us, UTC]
updated_at               datetime64[us, UTC]
closed_time              datetime64[us, UTC]
uma_resolution_status                    str
dtype: object


Bas

,parent_slug,market_id,question,condition_id,slug,end_date,closed,active,volume,volume_num,clob_token_id,outcome_prices,last_trade_price,best_bid,best_ask,group_item_title,created_at,updated_at,closed_time,uma_resolution_status
1354,elon-musk-of-tweets-january-23-january-30,1225160,Will Elon Musk post 360-379 tweets from Januar...,0xb41dbf64ca0c88f2fe9b19721211de85feeccc5806b4...,elon-musk-of-tweets-january-23-january-30-360-379,2026-01-30 17:00:00+00:00,True,True,1.714003e+06,1.714003e+06,7879574513179606895240225779527533095819429702...,"[0, 1]",0.001,NaN,0.001,360-379,2026-01-20 05:00:10.944348+00:00,2026-02-01 00:23:12.459243+00:00,2026-01-30 19:18:56+00:00,resolved
854,elon-musk-of-tweets-september-19-september-26,599885,Will Elon Musk post 520-539 tweets from Septem...,0x03ca27b16a1e4eabe4e14c9609a939a8283567b07839...,elon-musk-of-tweets-september-19-september-26-...,2025-09-26 16:00:00+00:00,True,True,5.536743e+04,5.536743e+04,7167677309264027955982041900780287499897282885...,"[0, 1]",1.000,NaN,0.001,520-539,2025-09-16 04:00:01.715222+00:00,2025-09-28 00:46:09.987780+00:00,2025-09-26 19:42:29+00:00,resolved
866,elon-musk-of-tweets-september-26-october-3,609120,Will Elon Musk post 280-299 tweets from Septem...,0x3f8fe640edebff0ad85a3550e0544e898a8d9ff6b0b2...,elon-musk-of-tweets-september-26-october-3-280...,2025-10-03 16:00:00+00:00,True,True,2.046728e+05,2.046728e+05,6745264541641242570897518378063986765039283386...,"[0, 1]",1.000,NaN,0.001,280-299,2025-09-23 04:00:01.841056+00:00,2025-10-05 00:38:07.449541+00:00,2025-10-03 19:24:36+00:00,resolved
762,elon-musk-of-tweets-august-29-september-5,583536,Will Elon tweet 325–339 times August 29–Septem...,0x9fc53eb2cdc3ca8b6ef2540538460e7643c52e9ef51a...,will-elon-tweet-325339-times-august-29september-5,2025-09-05 04:00:00+00:00,True,True,2.970820e+05,2.970820e+05,1149546920039386765750276540843269769346932222...,"[0, 1]",1.000,NaN,0.001,325–339,2025-08-29 15:18:36.988280+00:00,2025-09-04 01:08:26.341741+00:00,2025-09-02 21:22:00+00:00,resolved
908,elon-musk-of-tweets-october-3-october-10,617845,Will Elon Musk post 480-499 tweets from Octobe...,0xc7b3c50256f74843626c3ea0b4550edbec5f511b3c59...,elon-musk-of-tweets-october-3-october-10-480-499,2025-10-10 16:00:00+00:00,True,True,3.988128e+05,3.988128e+05,6066786863192655596706766612008051839139639145...,"[0, 1]",1.000,NaN,0.001,480-499,2025-09-30 04:00:09.845844+00:00,2025-10-11 08:40:30.534290+00:00,2025-10-10 19:33:04+00:00,resolved
